# Benchmark

> Benchmark module

In [ ]:
#| default_exp benchmark

In [ ]:
#| export
from __future__ import annotations

import json
import torch
from collections.abc import Mapping
from dataclasses import dataclass, field
from typing import Sequence, Dict, Any, Optional, Iterator

from fasterbench.size import SizeMetrics, compute_size
from fasterbench.speed import SpeedMetrics, compute_speed_multi
from fasterbench.compute import ComputeMetrics, compute_compute
from fasterbench.memory import MemoryMetrics, compute_memory_multi
from fasterbench.energy import EnergyMetrics, compute_energy_multi

In [ ]:
#| export
def _fmt_params(n: int) -> str:
    """Format parameter count with appropriate suffix."""
    if n >= 1e9: return f"{n/1e9:.2f}B"
    if n >= 1e6: return f"{n/1e6:.2f}M"
    if n >= 1e3: return f"{n/1e3:.2f}K"
    return str(n)

def _section(title: str) -> str:
    """Create a section header."""
    return f"═══ {title} " + "═" * (40 - len(title))


#| export
@dataclass
class BenchmarkResult(Mapping):
    """Structured container for benchmark results with typed access.
    
    Provides both typed attribute access (e.g., `result.speed["cpu"].mean_ms`)
    and backward-compatible dict-like access (e.g., `result["speed_cpu_mean_ms"]`).
    """
    size: Optional[SizeMetrics] = None
    speed: Dict[str, SpeedMetrics] = field(default_factory=dict)
    compute: Optional[ComputeMetrics] = None
    memory: Dict[str, MemoryMetrics] = field(default_factory=dict)
    energy: Dict[str, EnergyMetrics] = field(default_factory=dict)
    _dict_cache: Optional[Dict[str, Any]] = field(default=None, repr=False)

    def as_dict(self) -> Dict[str, Any]:
        """Return flat dict matching the original `benchmark()` return format. Cached."""
        if self._dict_cache is not None:
            return self._dict_cache
        
        out: Dict[str, Any] = {}
        if self.size is not None:
            out.update({f"size_{k}": v for k, v in self.size.as_dict().items()})
        for dev, met in self.speed.items():
            out.update({f"speed_{dev}_{k}": v for k, v in met.as_dict().items()})
        if self.compute is not None:
            out.update({f"compute_{k}": v for k, v in self.compute.as_dict().items()})
        for dev, met in self.memory.items():
            out.update({f"memory_{dev}_{k}": v for k, v in met.as_dict().items()})
        for dev, met in self.energy.items():
            out.update({f"energy_{dev}_{k}": v for k, v in met.as_dict().items()})
        
        object.__setattr__(self, '_dict_cache', out)
        return out

    def _format_summary(self) -> str:
        """Build the formatted summary string."""
        lines = []
        
        if self.size is not None:
            lines.append(_section("Size"))
            lines.append(f"  Disk:   {self.size.size_mib:.2f} MiB")
            lines.append(f"  Params: {_fmt_params(self.size.num_params)}")
        
        if self.speed:
            lines.append(_section("Speed"))
            for dev, m in self.speed.items():
                lines.append(f"  {dev}: {m.mean_ms:.2f} ms  │  {m.throughput_s:.1f} inf/s  │  p99: {m.p99_ms:.2f} ms")
        
        if self.compute is not None:
            lines.append(_section("Compute"))
            if self.compute.macs_available:
                lines.append(f"  MACs:   {self.compute.macs_m:.1f} M")
            lines.append(f"  Params: {self.compute.params_m:.2f} M")
        
        if self.memory:
            lines.append(_section("Memory"))
            for dev, m in self.memory.items():
                lines.append(f"  {dev}: peak {m.peak_mib:.2f} MiB  │  avg {m.avg_mib:.2f} MiB")
        
        if self.energy:
            lines.append(_section("Energy"))
            for dev, m in self.energy.items():
                lines.append(f"  {dev}: {m.mean_watts:.1f} W  │  {m.energy_wh:.4f} Wh/inf  │  {m.co2_eq_g:.4f} g CO₂/inf")
        
        return "\n".join(lines) if lines else "No metrics measured"

    def summary(self) -> None:
        """Print a human-readable summary of benchmark results."""
        print(self._format_summary())

    def to_dataframe(self):
        """Convert results to a pandas DataFrame."""
        import pandas as pd
        return pd.DataFrame([self.as_dict()])

    def to_json(self) -> str:
        """Serialize to JSON string."""
        return json.dumps(self.as_dict(), indent=2)

    # Mapping protocol - only 3 methods needed, rest inherited
    def __getitem__(self, key: str) -> Any: return self.as_dict()[key]
    def __iter__(self) -> Iterator[str]: return iter(self.as_dict())
    def __len__(self) -> int: return len(self.as_dict())

    def __str__(self) -> str: return self._format_summary()
    def __repr__(self) -> str: return self._format_summary()

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
_VALID_METRICS = frozenset({"size", "speed", "compute", "memory", "energy"})


def benchmark(
    model: torch.nn.Module,                                                      # model to profile
    sample: torch.Tensor,                                                        # input tensor (with batch dimension)
    *,
    metrics: Sequence[str] = ("size", "speed", "compute", "memory", "energy"),   # metrics to compute
    speed_devices: Sequence[str | torch.device] | None = None,                   # devices for speed (default: cpu + cuda)
    memory_devices: Sequence[str | torch.device] | None = None,                  # devices for memory
    energy_devices: Sequence[str | torch.device] | None = None,                  # devices for energy
    **kwargs,
) -> BenchmarkResult:
    """Run comprehensive benchmarks on a model."""
    metrics_set = set(metrics)
    if invalid := metrics_set - _VALID_METRICS:
        raise ValueError(f"Invalid metrics: {invalid}. Valid: {_VALID_METRICS}")
    if sample.dim() == 0:
        raise ValueError("sample must have at least 1 dimension")

    params = sum(p.numel() for p in model.parameters() if p.requires_grad) if metrics_set & {"size", "compute"} else None
    was_training = model.training
    model.eval()

    try:
        return BenchmarkResult(
            size=compute_size(model, params_count=params) if "size" in metrics_set else None,
            speed=compute_speed_multi(model, sample, devices=speed_devices, **kwargs) if "speed" in metrics_set else {},
            compute=compute_compute(model, sample, params_count=params) if "compute" in metrics_set else None,
            memory=compute_memory_multi(model, sample, devices=memory_devices, **kwargs) if "memory" in metrics_set else {},
            energy=compute_energy_multi(model, sample, devices=energy_devices, **kwargs) if "energy" in metrics_set else {},
        )
    finally:
        if was_training:
            model.train()

In [ ]:
show_doc(benchmark)

---

[source](https://github.com/FasterAI-Labs/fasterbench/blob/main/fasterbench/benchmark.py#L20){target="_blank" style="float:right; font-size:smaller"}

### benchmark

>      benchmark (model:torch.nn.modules.module.Module, sample:torch.Tensor,
>                 metrics:Sequence[str]=('size', 'speed', 'compute', 'memory',
>                 'energy'),
>                 speed_devices:Optional[Sequence[str|torch.device]]=None,
>                 memory_devices:Optional[Sequence[str|torch.device]]=None,
>                 energy_devices:Optional[Sequence[str|torch.device]]=None,
>                 **kwargs)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | Module |  | the model to profile (can stay on CPU) |
| sample | Tensor |  | dummy input of the right shape |
| metrics | Sequence | ('size', 'speed', 'compute', 'memory', 'energy') |  |
| speed_devices | Optional | None |  |
| memory_devices | Optional | None |  |
| energy_devices | Optional | None |  |
| kwargs |  |  |  |
| **Returns** | **Dict** |  |  |